In [ ]:
# ============================================
# doc_hint_matching_experiment_2.ipynb
#
# [실험 목적]
# doc_hint_matching_experiment_2에서 고친 것들(접미사 노이즈, fuzzy
# min_overlap) 말고 남아있는 오탐 케이스가 더 있는지 확인하고, core40
# 채점 함수가 기권 답변을 놓치는 문제도 같이 확인하는 게 목적
#
# [진행 방식과 알아낸 것]
#
# 1. 복합 키워드가 제대로 잡히는지 재검증 (cell 18~19)
#    - "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템
#      고도화"처럼 사업명이 긴 경우, "학사정보시스템"이라는 복합 키워드가
#      fuzzy_match(min_overlap=6) 조건에서 정확히 잡히는지 하나씩 확인
#
#
# 2. 블랙리스트를 우회하는 새로운 패턴 발견 (cell 21, 26)
#    - substring을 조금 더 길게 잡으면, 그 안에 블랙리스트 단어가
#      "포함된 채로" 매칭 조건은 통과해버리는 걸 발견 (예: "대학교
#      산학협력단"처럼 블랙리스트 단어 "대학교"가 더 긴 문자열 안에
#      숨어있는 경우)
#    - _check_contains_blacklist_word() 함수를 새로 만들어서, substring
#      안에 블랙리스트 단어가 통째로 하나라도 포함돼 있으면 그 자체로
#      걸러내도록 보강
#
#
# 3. 문서 1개뿐인 발주기관 오탐 재현 (cell 18과 연계)
#    - 한영대학·경희대학교처럼 코퍼스에 딱 문서가 1개뿐인 기관들이
#      고려대·서영대 등 다른 기관과 오매칭되는 걸 확인
#    - 위 [1][2]에서 고친 내용으로 이 오탐들도 같이 해결됨을 확인
#
#
# 4. core40 채점 함수(ABSTAIN_PHRASES) 보강 (cell 32, 35)
#    - normalize_text/normalize_dates/_text_included 등 core40 전용
#      채점 함수 전체를 이 노트북에 다시 정의
#    - 실제 답변을 훑어보다가 "판정을 해드릴 수 없습니다"처럼 기권
#      문구인데 그 사이에 "판정을"이라는 다른 단어가 끼어 있어서
#      기존 ABSTAIN_PHRASES 목록("판정할 수 없")과 정확히 안 맞아
#      매칭에 실패하던 사례를 발견
#    - "해드릴 수 없", "드릴 수 없"라는 표현 두 개를 목록에 추가한
#      official_score_core40_v2로 이 사례를 구제함
# ============================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/sprint-public-procurement-rag-assistant
!pwd

/content/sprint-public-procurement-rag-assistant
/content/sprint-public-procurement-rag-assistant


In [3]:
import sys
import types
import src.data_processing.chunking as real_chunking

import pickle

from src.retrieval.indexing import HybridIndex
from src.data_processing.chunking import Chunk

import src.config as config
from pathlib import Path
import src.retrieval.indexing as indexing_module

config.CHROMA_DIR = Path('/content/drive/MyDrive/중급 프로젝트/chroma_db')

indexing_module.CHROMA_DIR = config.CHROMA_DIR
chunking_alias = types.ModuleType('src.chunking')
chunking_alias.Chunk = real_chunking.Chunk
sys.modules['src.chunking'] = chunking_alias

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

index = HybridIndex(chunks)

[HybridIndex] parent 전략 chunk 3664개는 검색 후보에서 제외(context 확장 조회 전용) - 실제 검색 대상 14575개
[embeddings] SentenceTransformer 모델 로드 시도 중... (처음 실행이면 HuggingFace에서 모델을 내려받아 몇 분 걸릴 수 있습니다)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

[embeddings] SentenceTransformer 사용: nlpai-lab/KURE-v1 (dim=1024)
[HybridIndex] 기존 임베딩 인덱스 재사용: output/chroma_db (collection=rfp_chunks__nlpai-lab_KURE-v1, backend=nlpai-lab/KURE-v1, 검색 대상 chunk 14575개 일치, 재임베딩 건너뜀)


In [4]:
child_chunks = index._searchable_chunks
print(f"검색 대상(child) chunk 수: {len(child_chunks)}")

검색 대상(child) chunk 수: 14575


In [5]:
from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

In [6]:
seen = set()
all_filenames_with_biz = []
for c in child_chunks:
    if c.doc_id not in seen:
        seen.add(c.doc_id)
        biz = c.metadata.get('발주_기관', '')
        all_filenames_with_biz.append((c.doc_id, biz))

print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [7]:
sys.path.append('/content/drive/MyDrive/중급 프로젝트')

from generation_prompts import SYSTEM_PROMPT_V9, METADATA_DISTINCTION_INSTRUCTION, needs_metadata_distinction
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

print("import 성공")

import 성공


In [9]:
import json
from src.generation.generation import check_required_facts

DATA_DIR2 = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR2 / 'dev.refined.review-candidate.jsonl', encoding='utf-8') as f:
    core40 = [json.loads(l) for l in f if l.strip()]
with open(DATA_DIR2 / 'rag-56.draft.jsonl', encoding='utf-8') as f:
    rag56 = [json.loads(l) for l in f if l.strip()]
print(f"core40: {len(core40)}개, rag56: {len(rag56)}개")

core40: 40개, rag56: 56개


In [10]:
# 전체 98개 문서 파일명에서 자주 등장하는 단어 빈도 확인
from collections import Counter
import re

word_counter = Counter()
for fname, biz in all_filenames_with_biz:
    fname_clean = fname.replace('.hwp', '').replace('.pdf', '')
    words = re.split(r'[\s_·\(\)]+', fname_clean)
    for w in words:
        w = w.strip()
        if len(w) >= 4:
            word_counter[w] += 1

# 여러 문서 파일명에 등장하는 단어들 (오탐 위험군)
risky_words = {w: c for w, c in word_counter.items() if c >= 3}
print(f"3개 이상 문서에 등장하는 4글자+ 단어: {len(risky_words)}개")
for w, c in sorted(risky_words.items(), key=lambda x: -x[1]):
    print(f"  '{w}': {c}회")

3개 이상 문서에 등장하는 4글자+ 단어: 12개
  '2024년': 13회
  '기능개선': 7회
  '정보시스템': 7회
  '홈페이지': 4회
  '인천광역시': 3회
  '구축사업': 3회
  '한국수자원공사': 3회
  '한국철도공사': 3회
  '수협중앙회': 3회
  '2025년': 3회
  '전산시스템': 3회
  '재단법인': 3회


In [11]:
# 위험 단어들이 실제로 서로 다른 발주기관 문서를 오염시키는지 확인
risky_test_words = ['기능개선', '정보시스템', '홈페이지', '구축사업', '전산시스템']

for word in risky_test_words:
    matching_docs = [f for f, biz in all_filenames_with_biz if word in f]
    print(f"'{word}' 포함 문서 ({len(matching_docs)}개):")
    for d in matching_docs:
        print(f"  {d}")
    print()

'기능개선' 포함 문서 (7개):
  한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp
  재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
  울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp
  부산관광공사_경영정보시스템 기능개선.hwp
  광주과학기술원_학사시스템 기능개선 사업.hwp
  축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp
  한국보육진흥원_연차별 자율 품질관리 시스템 기능개선.hwp

'정보시스템' 포함 문서 (19개):
  고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
  한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고.hwp
  경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
  한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp
  한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp
  인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp
  울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp
  부산관광공사_경영정보시스템 기능개선.hwp
  한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
  국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp
  중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp
  한국한의학연구원_통합정보시스템 고도화 용역.hwp
  경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp
  한국건강가정진흥원_2025년 아이돌봄인력 인적성 검사 정보시스템 운영.hwp
  남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp
  재단법인 한국장애인문화예술원_2024년 장애인문화예술정보시스템 이음.hwp
  경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp
  한국산업단지공단_산단 안전정보시스

In [12]:
# 위험 단어들이 실제 core40/rag-56 질문에 등장하는지, 그리고 문서 힌트가 오염되는지 확인
risky_words = ['기능개선', '정보시스템', '홈페이지', '구축사업', '전산시스템']

all_questions = [(it['case_id'], it['question'], 'core40') for it in core40] + \
                [(it['case_id'], it['question'], 'rag56') for it in rag56]

for cid, q, src in all_questions:
    for word in risky_words:
        if word in q:
            hints = extract_doc_hints_multi(q, all_filenames_with_biz)
            print(f"[{src}/{cid}] (걸린 단어: '{word}')")
            print(f"  질문: {q}")
            print(f"  힌트 개수: {len(hints)}")
            if len(hints) > 3:
                print(f"  힌트: {hints}")
            print()

[core40/dev-single-006] (걸린 단어: '전산시스템')
  질문: 종량제봉투 판매관리 전산시스템 개선사업에서 구현해야 할 핵심 기능과 기존 데이터 이관 범위는?
  힌트 개수: 3

[core40/dev-multi-006] (걸린 단어: '정보시스템')
  질문: 산단안전정보시스템과 차세대 응급의료 상황관리시스템은 대응 대상·핵심 기능·예산이 어떻게 다른가요?
  힌트 개수: 2

[rag56/supplemental-qa-c12] (걸린 단어: '구축사업')
  질문: GKL 그룹웨어 시스템 구축사업은 하도급이 가능한가요? 공동수급 조건도 알려주세요.
  힌트 개수: 1

[rag56/supplemental-qa-c14] (걸린 단어: '정보시스템')
  질문: 평택시 2024년도 버스정보시스템(BIS) 구축사업에서 기술능력 평가점수가 해당 평가분야 배점한도의 85% 미만이면 협상대상이 될 수 있나요?
  힌트 개수: 2

[rag56/supplemental-qa-c14] (걸린 단어: '구축사업')
  질문: 평택시 2024년도 버스정보시스템(BIS) 구축사업에서 기술능력 평가점수가 해당 평가분야 배점한도의 85% 미만이면 협상대상이 될 수 있나요?
  힌트 개수: 2

[rag56/supplemental-qa-c19] (걸린 단어: '구축사업')
  질문: 다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다.
  힌트 개수: 12
  힌트: ['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]

In [13]:
q = "종량제봉투 판매관리 전산시스템 개선사업에서 구현해야 할 핵심 기능과 기존 데이터 이관 범위는?"
hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(hints)

['파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp', '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp', '세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp']


In [14]:
answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

핵심 기능(구현해야 할 주요 기능)
- 전산관리 시스템: 주문·판매내역 조회, 재고관리, 결제내역 관리, 보고서·통계 출력(지역별·규모별·주문패턴별 등).  
- 인터넷 주문 연동: 온라인 주문, 주문내역 확인 기능 및 다양한 디바이스(PC/모바일/태블릿) 대응(반응형웹).  
- 결제시스템 개발/연동: 카드 및 계좌이체(가상계좌) 결제 지원(PG사 API 연동 등).  
- 관리자 기능 통합: 기존 관리자 기능(권한관리, 로그관리 등) 정상 동작 보장하며 기존 홈페이지와 통합 구축.  
- 지정판매소 위치조회 서비스: 시민이 최근 지정판매소의 판매내역을 조회 가능한 위치조회 기능.  
- 보안·품질 요건: 전자정부표준프레임워크 등 도입, 오픈소스 공통서비스 활용, 시큐어 코딩 등 웹개발 보안지침 준수.  
- 운영·분석 기능: 주문·판매 데이터 기반 통계·빅데이터 분석으로 재고·수요 예측 및 발주 편의성 향상.  
- 사용자 교육 및 전환 지원: 파주시청에서도 운영 가능하도록 기술이전·사용자 교육 등 제공.

기존 데이터 이관 범위
- 기존(노후화된) 운영 중인 판매관리시스템의 판매소 및 판매자료 전(全) 이관: 판매관리시스템에 저장된 판매소 정보와 판매기록(판매자료)을 새 시스템으로 이관해야 함.  
(문서상 표현: "노후화된 기 운영 시스템의 판매소 및 판매자료 이관")  

근거: 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp


In [15]:
# 문서가 1개뿐인 발주기관들 목록 확보
single_doc_orgs = {}
for fname, biz in all_filenames_with_biz:
    org_part = fname.split('_')[0].strip()
    single_doc_orgs.setdefault(org_part, []).append(fname)

single_doc_orgs = {org: fnames for org, fnames in single_doc_orgs.items() if len(fnames) == 1}
print(f"문서 1개뿐인 발주기관 수: {len(single_doc_orgs)}개")

문서 1개뿐인 발주기관 수: 74개


In [16]:
# core40/rag-56 질문에서 실제로 언급되는 (문서 1개뿐인) 발주기관 찾기
all_questions_text = [it['question'] for it in core40] + [it['question'] for it in rag56]

mentioned_single_orgs = []
for org in single_doc_orgs.keys():
    org_clean = org.replace('(사)', '').replace('(재)', '').strip()
    for q in all_questions_text:
        if org_clean and len(org_clean) >= 3 and org_clean in q:
            mentioned_single_orgs.append(org)
            break

print(f"질문에 언급되는 발주기관: {len(mentioned_single_orgs)}개")
for org in mentioned_single_orgs:
    print(f"  {org}: {single_doc_orgs[org]}")

질문에 언급되는 발주기관: 7개
  한영대학: ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp']
  서울시립대학교: ['서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf']
  경희대학교: ['경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp']
  울산광역시: ['울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp']
  부산관광공사: ['부산관광공사_경영정보시스템 기능개선.hwp']
  서민금융진흥원: ['서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp']
  국립인천해양박물관: ['국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp']


In [17]:
verify_single_orgs = [
    ("한영대학", "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?"),
    ("서울시립대학교", "CSV에는 서울시립대학교 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역의 사업금액이 0원으로 되어 있습니다. 원문 기준 실제 사업비는 얼마인가요?"),
    ("경희대학교", "경희대학교 산학협력단 정보시스템 운영 용역의 관련 문의 전화번호와 이메일은 무엇인가요?"),
    ("울산광역시", "울산광역시 버스정보시스템 확대 구축 및 기능개선 사업의 계약방법은 무엇인가요?"),
    ("부산관광공사", "부산관광공사의 '경영정보시스템 기능개선'은 공사·물품·용역 중 어떤 유형인가요?"),
    ("서민금융진흥원", "서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?"),
    ("국립인천해양박물관", "국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?"),
]

for org, q in verify_single_orgs:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{org}] 힌트 개수: {len(hints)}")
    print(f"  {hints}")
    print()

[한영대학] 힌트 개수: 2
  ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp', '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf']

[서울시립대학교] 힌트 개수: 1
  ['서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf']

[경희대학교] 힌트 개수: 2
  ['경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp', '서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp']

[울산광역시] 힌트 개수: 1
  ['울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp']

[부산관광공사] 힌트 개수: 1
  ['부산관광공사_경영정보시스템 기능개선.hwp']

[서민금융진흥원] 힌트 개수: 1
  ['서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp']

[국립인천해양박물관] 힌트 개수: 1
  ['국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp']



In [19]:
from answer_generation import COMMON_FILENAME_WORDS, COMMON_SUFFIX_WORDS
print('시스템' in COMMON_FILENAME_WORDS, '개량' in COMMON_FILENAME_WORDS)

True True


In [20]:
q1 = "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?"
q2 = "경희대학교 산학협력단 정보시스템 운영 용역의 관련 문의 전화번호와 이메일은 무엇인가요?"

# 3단계 키워드 확인
stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}

for q, target_fname in [(q1, '고려대학교_차세대 포털·학사 정보시스템 구축사업'), (q2, '서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육')]:
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q) if len(w) >= 4]
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

    def fuzzy_match(kw, text, min_overlap=6):
        kw_ns = kw.replace(' ', '')
        text_ns = text.replace(' ', '')
        if kw_ns in text_ns:
            return True
        for n in range(len(kw_ns), min_overlap - 1, -1):
            if kw_ns[:n] in text_ns:
                return True
        return False

    matched = [kw for kw in keywords_all if fuzzy_match(kw, target_fname)]
    print(f"질문 키워드: {keywords_all}")
    print(f"{target_fname}: matched={matched}")
    print()

질문 키워드: ['한영대학교', '교육환경', '트랙운영', '학사정보시스템', '발주기관은', '어디인가요']
고려대학교_차세대 포털·학사 정보시스템 구축사업: matched=['학사정보시스템']

질문 키워드: ['경희대학교', '정보시스템', '전화번호와', '이메일은']
서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육: matched=[]



In [21]:
# "학사정보시스템" - min_overlap 확인
def fuzzy_match(kw, text, min_overlap=6):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

print(fuzzy_match('학사정보시스템', '고려대학교_차세대 포털·학사 정보시스템 구축사업'))

# 경희대학교 케이스 - 1단계(기관명) 매칭 확인
q2 = "경희대학교 산학협력단 정보시스템 운영 용역의 관련 문의 전화번호와 이메일은 무엇인가요?"
target_org = '서영대학교 산학협력단'

min_len = 6
for start in range(len(target_org) - min_len + 1):
    for length in range(len(target_org) - start, min_len - 1, -1):
        substr = target_org[start:start+length]
        stripped = substr.strip()
        if len(stripped) >= min_len and stripped in q2 and stripped not in COMMON_SUFFIX_WORDS:
            print(f"1단계 매칭: '{stripped}'")

True
1단계 매칭: '대학교 산학협력단'
1단계 매칭: '대학교 산학협력'
1단계 매칭: '대학교 산학협'
1단계 매칭: '대학교 산학'
1단계 매칭: '학교 산학협력단'
1단계 매칭: '학교 산학협력'
1단계 매칭: '학교 산학협'
1단계 매칭: '교 산학협력단'
1단계 매칭: '교 산학협력'


In [22]:
# "대학교"를 COMMON_SUFFIX_WORDS에 이미 넣었는데, 왜 "교 산학협력단"이 걸렸는지 확인
print('대학교' in COMMON_SUFFIX_WORDS)

True


In [23]:
def _check_contains_blacklist_word(substr, blacklist):
    """substr 안에 블랙리스트 단어가 하나라도 통째로 포함되어 있으면 True"""
    for word in blacklist:
        if len(word) >= 3 and word in substr:
            return True
    return False

test_substrs = ['대학교 산학협력단', '대학교 산학협력', '학교 산학협력단']
for s in test_substrs:
    result = _check_contains_blacklist_word(s, COMMON_SUFFIX_WORDS)
    print(f"'{s}' -> 블랙리스트 단어 포함: {result}")

'대학교 산학협력단' -> 블랙리스트 단어 포함: True
'대학교 산학협력' -> 블랙리스트 단어 포함: True
'학교 산학협력단' -> 블랙리스트 단어 포함: True


In [24]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

idx = content.find("            min_len = 6")
print(content[idx:idx+600])

            min_len = 6
            for target_str in [org_core_clean, org_core_norm]:
                for start in range(len(target_str) - min_len + 1):
                    for length in range(len(target_str) - start, min_len - 1, -1):
                        substr = target_str[start:start+length]
                        stripped_substr = substr.strip()
                        if len(stripped_substr) >= min_len and stripped_substr in question and stripped_substr not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break
                    if mat


In [25]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "if len(stripped_substr) >= min_len and stripped_substr in question and stripped_substr not in COMMON_SUFFIX_WORDS:\n                            matched = True\n                            break"
new_code = "contains_blacklist = any(len(w) >= 3 and w in stripped_substr for w in COMMON_SUFFIX_WORDS)\n                        if len(stripped_substr) >= min_len and stripped_substr in question and stripped_substr not in COMMON_SUFFIX_WORDS and not contains_blacklist:\n                            matched = True\n                            break"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [26]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

test_cases = [
    "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?",
    "경희대학교 산학협력단 정보시스템 운영 용역의 관련 문의 전화번호와 이메일은 무엇인가요?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?]
  힌트: ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp', '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf']

[경희대학교 산학협력단 정보시스템 운영 용역의 관련 문의 전화번호와 이메일은 무엇인가요?]
  힌트: ['경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp']



In [27]:
# "학사정보시스템"이 COMMON_SUFFIX_WORDS의 어떤 단어를 포함하는지 확인
contains = [w for w in COMMON_SUFFIX_WORDS if len(w) >= 3 and w in '학사정보시스템']
print(contains)

['시스템']


In [28]:
def fuzzy_match_test(kw, text, min_overlap=6):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

def contains_blacklist_word(kw, blacklist):
    return any(len(w) >= 3 and w in kw for w in blacklist)

# core40/rag-56 전체 질문에서, "블랙리스트 단어를 포함한 키워드"가 실제로 매칭에 기여하는 비율 확인
affected_count = 0
total_keyword_matches = 0

for it in core40 + rag56:
    q = it['question']
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q) if len(w) >= 4]
    stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

    for kw in keywords_all:
        if contains_blacklist_word(kw, COMMON_SUFFIX_WORDS):
            affected_count += 1
        total_keyword_matches += 1

print(f"블랙리스트 단어를 포함한 키워드 비율: {affected_count}/{total_keyword_matches}")

블랙리스트 단어를 포함한 키워드 비율: 90/491


In [29]:
# "학사정보시스템"을 COMMON_FILENAME_WORDS에 추가하되, 이게 실제로 안전한지 먼저 확인
# (다른 곳에서 이 단어가 꼭 필요한 매칭이었는지)
affected_questions = []
for it in core40 + rag56:
    if '학사정보시스템' in it['question'] or '학사 정보시스템' in it['question']:
        affected_questions.append(it['question'])

print(f"'학사정보시스템' 관련 질문: {len(affected_questions)}개")
for q in affected_questions:
    print(f"  {q}")

'학사정보시스템' 관련 질문: 2개
  '한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화' 사업의 발주기관은 어디인가요?
  한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?


In [30]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역', '개량', 'ISMP', '정보화사업', '정보화'}"
new_code = "COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역', '개량', 'ISMP', '정보화사업', '정보화', '학사정보시스템', '학사 정보시스템'}"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [31]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_doc_hints_multi

importlib.reload(answer_generation)

test_cases = [
    "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?",
    "한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?",
]

for q in test_cases:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?]
  힌트: ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp']

[한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?]
  힌트: ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp']



In [32]:
verify_answers_all = [
    ("한영대학", "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?"),
    ("한영대학", "한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?"),
    ("서울시립대학교", "CSV에는 서울시립대학교 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역의 사업금액이 0원으로 되어 있습니다. 원문 기준 실제 사업비는 얼마인가요?"),
    ("경희대학교", "경희대학교 산학협력단 정보시스템 운영 용역의 관련 문의 전화번호와 이메일은 무엇인가요?"),
    ("울산광역시", "울산광역시 버스정보시스템 확대 구축 및 기능개선 사업의 계약방법은 무엇인가요?"),
    ("부산관광공사", "부산관광공사의 '경영정보시스템 기능개선'은 공사·물품·용역 중 어떤 유형인가요?"),
    ("서민금융진흥원", "서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?"),
    ("국립인천해양박물관", "국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?"),
]

for org, q in verify_answers_all:
    print(f"[{org}] {q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

[한영대학] 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 발주기관은 어디인가요?
발주기관: 한영대학 (한영대학교) — 근거: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp

[한영대학] 한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?
제안서 10부(원본 1부 + 사본 9부) 및 제안내용 수록 USB 1매. 근거: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 제안요청서 (한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp).

[서울시립대학교] CSV에는 서울시립대학교 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역의 사업금액이 0원으로 되어 있습니다. 원문 기준 실제 사업비는 얼마인가요?
금242,900,000원(부가가치세 포함). 근거: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf

[경희대학교] 경희대학교 산학협력단 정보시스템 운영 용역의 관련 문의 전화번호와 이메일은 무엇인가요?
전화번호: 031-201-3560
이메일: khcho@khu.ac.kr

근거: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp

[울산광역시] 울산광역시 버스정보시스템 확대 구축 및 기능개선 사업의 계약방법은 무엇인가요?
계약방법: 제한경쟁입찰(협상에 의한 계약). 근거: 울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp

[부산관광공사] 부산관광공사의 '경영정보시스템 기능개선'은 공사·물품·용역 중 어떤 유형인가요?
용역(서비스)입니다.

근거: 부산관광공사_경영정보시스템 기능개선.hwp

[서민금융진흥원] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출 방식: 입찰서와 제안서는 나라장터를 통한 전자제출(전자입찰)로만 제출해야 하며, 입찰서와 제안서를 모두 제출해야 유

In [34]:
# core40 전체 답변 생성
final_check_40 = []
for item in core40:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_check_40.append({'case_id': item['case_id'], 'task_type': task_type, 'answer': answer})
    print(f"[{item['case_id']}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 금 243,000,000원 — 부가가치세(VAT) 포함.  
근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원 (VAT 포함)

근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차로 구분됨: 1차(시스템 구축 및 초기 데이터 구축), 2차(리포팅툴 S/W 및 리포트 출력양식 개발).  
- 평가 비중: 기술평가 90%, 가격평가 10%.

근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
- 사업기간: 계약일로부터 90일(약 3개월). (추진일정상 계약일로부터 3개월, 최종 테스트 지원일: ’24.11.9)  
- 시범 도입 규모: 1단계 3개 기관(서울 2개소, 울산 1개소).

근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입.hwp

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 입찰서·제안서 제출 방식: 입찰서는 나라장터를 통해 전자적으로만 제출해야 하며(입찰서와 제안서를 모두 제출해야 유효). 기타 제출서류는 나라장터(e-발주시스템)를 통한 전자제출 또는 서민금융진흥원 IT전략부 직접

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다. 제공된 문서 범위에서는 '오늘' 올라온 나라장터 공고(실시간 정보)를 확인할 수 없습니다.

[dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다. 근거: 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp

[dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
이 요청은 문서와 사용자의 회사 정보를 대조해 객관적·주관적 판단을 내려야 하므로 본 규칙에 따라 판정(적격/부적격)을 해드릴 수 없습니다. 확인에 필요한 조건 목록만 제공하겠습니다.

다음 항목들을 귀사 내부 자료로 확인하세요(문서 근거 항목 포함).

필수 확인 항목(문서 근거)
1. 부정당업자 해당 여부: 「지방자치단체를 당사자로 하는 계약에 관한 법률 시행령 제92조」에 해당되지 않는지 확인.  
   근거: 입찰참가자격 가. (문서)
2. 주된 영업소 소재지: 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역시인지 확인(관련 법령 준수).  
   근거: 입찰참가자격 나. (문서)
3. 나라장터(G2B) 등록: 입찰서 제출마감일 전일까지 소프트웨어사업자(업종코드 1468, 컴퓨터관련서비스사업)로 입찰참가자격 등록되어 있는지 확인.  
   근거: 입찰참가자격 다. (문서)
4. 기업 규모 제한: 소프트웨어산업 진흥법 제48조 및 중소소프트웨어사업자 지원지침에 따라 대기업·중견기업 및 상호출자제한기업집단 소속회사가 아닌지 확인.  
   근거: 입찰참가자격 라. (문서)
5. 직접생산확인증명서: 정보시스템개발서비스(세부품명번호 8111159901)에 대한 ‘직접생산확인증명서’를 입찰마감 전일까지 발급받아 유효기간 내인지 확인.  
   근거: 입찰참가자격 마. (문서)
6. 공동수급·하도급 금지: 공동수급(공동이행방식) 불허, 하도급 불허 조건을 준수할 것(단독 참여 가능 여부 확인).  
   근거: 입찰참가자격 사. 및 별도 문구(하도급 불허) (문서)

추가 유의사항
- 각 항

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
확인되지 않습니다.

요청하신 내용(평가위원들의 실제로 더 중요하게 보는 비공개 평가 포인트)은 제공된 문서 범위 내에 근거할 수 있는 사실이 아니므로 답변할 수 없습니다.

참고로, 이런 판단을 위해 문서에서 필요할 수 있는 근거(사용자가 내부적으로 확인해 주면 판단에 도움이 되는 항목)는 간단히 안내드립니다.
- 제안평가의 세부 평가항목별 배점표(공개된 배점 외에 위원별 가중치·세부기준)
- 과거 평가결과(평가위원별 채점 분포·평가 코멘트)
- 평가위원 구성(전문성·관심분야) 및 평가위원회의 내부지침
- 심사 당시의 질의응답·발표 녹취 내용 및 추가자료 요청 내역

근거 문서: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp 등 (제공된 문서 전체)

[dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
확인되지 않습니다.

입찰 수주 확률(%) 산정은 제안사 역량·가격·평가결과·경쟁사 현황 등 문서에 없는 정보와 판단을 요구하므로 제공된 문서만으로는 계산할 수 없습니다. 이 기능은 제가 수행할 수 없습니다.

필요하면 수주확률을 판단하는 데 일반적으로 요구되는 핵심 정보(간단히):
- 제안 평가 기준별 배점·평가기준(세부 점수화 방법)  
- 우리 회사의 기술·실적·인력·가격 제안 내용(제안서 초안)  
- 경쟁사 후보군과 그들의 예상 제안 수준  
- 입찰참가 자격 충족 여부, 입찰보증금·계약조건 등 리스크 요인  
- 발주처의 우선순위(지역제한·협상방식 등 발주조건)

위 정보들을 문서로 제공해주시면, 문서 범위 안에서 판단에 필요한 항목을 정리해 드릴 수 있습니다.

근거 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
문서에 없는 정보를 임의로 가정해서 확정할 수 없습니다. 대신 원하시면 제공된 문서들에서 '공개일(공고일)'을 찾아 드리겠습니다. 근거: 제공된 문서들

[dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
확인할 수 없습니다.

제공된 문서 범위에는 현재 진행률(%)이나 계약 지연 가능성에 대한 정보가 없습니다. 이런 판단을 하려면 다음과 같은 근거자료가 필요합니다(참고용 최소 항목):
- 현재까지 완료된 산출물·마일스톤 현황(진행보고서)
- 계약 체결일 및 실제 착수일 비교 자료
- 일정 지연 원인(인수·인계, 추가요구, 자원파견 지연 등)과 변경(변경명세서)
- 인력·장비·외부업체 투입 현황 및 가용성
- 발주처(부산국제영화제)와의 승인·검수 진행상태

원문 근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp



In [35]:
def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]

    if numbers:
        all_numbers_match = True
        for num in numbers:
            if num in answer_norm:
                continue
            if len(num) == 4 and num.startswith('20'):
                if num[2:] in answer_norm:
                    continue
            num_no_zero = re.sub(r'^0+', '', num)
            if num_no_zero and num_no_zero in answer_norm:
                continue
            all_numbers_match = False
            break

        if not all_numbers_match:
            return False

        raw_words = re.split(r'[\s,·:()]+', fact_text)
        stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
        core_words = []
        for w in raw_words:
            w = w.rstrip('.,')
            if len(w) < 2:
                continue
            if re.match(r'^\d', w):
                continue
            for suf in stopwords_suffix:
                if w.endswith(suf) and len(w) > len(suf):
                    w = w[:-len(suf)]
                    break
            w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
            if len(w) >= 2:
                core_words.append(w)

        if not core_words:
            return True

        match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
        return match_count / max(len(core_words), 1) >= 0.2

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는']

def official_score_core40(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [36]:
for r in final_check_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in final_check_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

by_type = {}
for r in final_check_40:
    if r['score'] is not None:
        by_type.setdefault(r['task_type'], []).append(r['score'])
for t, scores in by_type.items():
    print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100 ({len(scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 75.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 75.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 75.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 50.0
[dev-f

In [37]:
answer = next(r['answer'] for r in final_check_40 if r['case_id'] == 'dev-unknown-004')
print(answer[:300])
print()

for phrase in ABSTAIN_PHRASES:
    if phrase in answer:
        print(f"발견된 기권 문구: '{phrase}'")

이 요청은 문서와 사용자의 회사 정보를 대조해 객관적·주관적 판단을 내려야 하므로 본 규칙에 따라 판정(적격/부적격)을 해드릴 수 없습니다. 확인에 필요한 조건 목록만 제공하겠습니다.

다음 항목들을 귀사 내부 자료로 확인하세요(문서 근거 항목 포함).

필수 확인 항목(문서 근거)
1. 부정당업자 해당 여부: 「지방자치단체를 당사자로 하는 계약에 관한 법률 시행령 제92조」에 해당되지 않는지 확인.  
   근거: 입찰참가자격 가. (문서)
2. 주된 영업소 소재지: 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역



In [38]:
ABSTAIN_PHRASES_V2 = ABSTAIN_PHRASES + ['해드릴 수 없', '드릴 수 없']

def official_score_core40_v2(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES_V2)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

score = official_score_core40_v2(next(it for it in core40 if it['case_id'] == 'dev-unknown-004'), answer)
print(f"수정된 채점: {score}")

수정된 채점: 100


In [39]:
for r in final_check_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40_v2(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in final_check_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

by_type = {}
for r in final_check_40:
    if r['score'] is not None:
        by_type.setdefault(r['task_type'], []).append(r['score'])
for t, scores in by_type.items():
    print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100 ({len(scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 75.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 75.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 75.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 50.0
[dev-f

In [40]:
item = next(it for it in core40 if it['case_id'] == 'dev-followup-004')
print("정답 요소:")
for kp in item['gold']['required_key_points']:
    print(f"  - {kp['text']}")
print()
answer = next(r['answer'] for r in final_check_40 if r['case_id'] == 'dev-followup-004')
print("답변:", answer)

정답 요소:
  - 심층 문의는 전화상담으로 제공한다.
  - 채팅과 1397콜센터 전화상담 간 채널전환 기능을 구현한다.

답변: 간편·반복적인 문의는 채팅상담(및 챗봇)으로 우선 처리하려고 합니다. 심층 문의는 전화상담으로 전환하여 처리합니다. 근거: 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp


In [41]:
# 파일명에 영문 약어가 포함된 문서들 찾기
import re

eng_abbrev_docs = []
for fname, biz in all_filenames_with_biz:
    eng_matches = re.findall(r'[A-Z]{2,}', fname)
    if eng_matches:
        eng_abbrev_docs.append((fname, eng_matches))

print(f"영문 약어 포함 문서: {len(eng_abbrev_docs)}개")
for fname, abbrevs in eng_abbrev_docs:
    print(f"  {fname[:60]} -> {abbrevs}")

영문 약어 포함 문서: 32개
  한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp -> ['UICC']
  한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp -> ['EIP']
  재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp -> ['GIS']
  재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp -> ['LMS']
  (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp -> ['KUSF']
  한국수자원공사_건설통합시스템(CMS) 고도화.hwp -> ['CMS']
  한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp -> ['ISMP']
  전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp -> ['JST', 'API', 'LRS']
  한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp -> ['AFSIS']
  KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp -> ['KOICA']
  인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp -> ['ISP']
  대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp -> ['MILE']
  한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp -> ['DB']
  한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp -> ['ERP']
  대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp -> ['APC', 'HUB']
  한국어촌어항공단_한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도.hwp -> ['ERP', 'GW']
  (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp -> ['BIFF', 'A

In [42]:
isp_questions = ["인천일자리플랫폼 ISP 사업 예산은?", "국가철도공단 철도인프라 디지털트윈 ISP 수립 용역 평가 배점은?", "서울교육청 ISP 사업기간은?"]
erp_questions = ["한국가스공사 차세대 통합정보시스템(ERP) 구축 사업 예산은?", "인천공항운영서비스 차세대 ERP시스템 구축 사업 기간은?"]

for q in isp_questions + erp_questions:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[인천일자리플랫폼 ISP 사업 예산은?]
  힌트: ['인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp']

[국가철도공단 철도인프라 디지털트윈 ISP 수립 용역 평가 배점은?]
  힌트: ['국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp']

[서울교육청 ISP 사업기간은?]
  힌트: ['서울특별시교육청_서울특별시교육청 지능정보화전략계획(ISP) 수립(2차) .hwp']

[한국가스공사 차세대 통합정보시스템(ERP) 구축 사업 예산은?]
  힌트: ['한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국한의학연구원_통합정보시스템 고도화 용역.hwp', '(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp']

[인천공항운영서비스 차세대 ERP시스템 구축 사업 기간은?]
  힌트: ['인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp']



In [43]:
answer = ask_rfp_v9("한국가스공사 차세대 통합정보시스템(ERP) 구축 사업 예산은?", client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

14,107,009,000원 — 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp


In [44]:
verify_answers_2_all = [
    "인천일자리플랫폼 ISP 사업 예산은?",
    "국가철도공단 철도인프라 디지털트윈 ISP 수립 용역 평가 배점은?",
    "서울교육청 ISP 사업기간은?",
    "한국가스공사 차세대 통합정보시스템(ERP) 구축 사업 예산은?",
    "인천공항운영서비스 차세대 ERP시스템 구축 사업 기간은?",
]

for q in verify_answers_2_all:
    print(f"{q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

인천일자리플랫폼 ISP 사업 예산은?
금100,000천원(금일억원) — 100,000,000원. 근거: 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp

국가철도공단 철도인프라 디지털트윈 ISP 수립 용역 평가 배점은?
기본 배점
- 기술평가: 90%
- 가격평가: 10%
(근거: 평가요소(100점): 기술평가(90%), 가격평가(10%)) — 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp)

제안서 기술능력(세부 배점, 기술평가 내 배점 합계 100점 기준)
- 사업수행능력: 15점
  - 경영상태: 5점
  - 수행경험 및 실적: 5점
  - 투입인력 적정성: 5점
- 전략 및 방법론: 30점
  - 사업이해도: 10점
  - 추진전략: 10점
  - 추진방법론: 10점
- 기술 및 기능: 30점
  - 컨설팅 요구사항: 10점
  - 데이터 요구사항: 5점
  - 보안 요구사항: 5점
  - 품질 요구사항: 5점
  - 제약사항: 5점
- 프로젝트 관리: 15점
  - 관리 방법론: 5점
  - 일정계획: 5점
  - 수행조직 및 이력사항: 5점
- 프로젝트 지원: 10점
  - 기밀보안: 5점
  - 비상대책: 5점

기타 관련 사항
- 비계량(등급) 평가는 등급별 배점표 및 참여업체 수에 따른 상대배점 규정이 별도로 적용됨.  
(근거 문서: 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp))

서울교육청 ISP 사업기간은?
계약일로부터 5개월입니다. 근거: 서울특별시교육청_서울특별시교육청 지능정보화전략계획(ISP) 수립(2차) .hwp

한국가스공사 차세대 통합정보시스템(ERP) 구축 사업 예산은?
사업금액: 14,107,009,000원

근거: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp

인천공항운영서비스 차세대 ERP시스템 구축 사업 기간은?
계약 체결일로부터 9개월(안정화 기간 포함). 근거: 인천공항운영서비스㈜ 차세대 ERP시스템 구축 제안요

In [45]:
different_expression_tests = [
    "국방과학연구소가 파일 전송하는 시스템을 크게 개선하려는 사업 예산은?",
    "국방과학연구소가 문서나 자료를 체계적으로 관리하고 보안을 강화하려는 사업 예산은?",
    "축산물품질평가원이 동물 이력을 추적하는 시스템을 손보는 사업 기간은?",
    "축산물품질평가원의 벌꿀 등급 판정 관련 시스템 개선 사업 예산은?",
]

for q in different_expression_tests:
    hints = extract_doc_hints_multi(q, all_filenames_with_biz)
    print(f"[{q}]")
    print(f"  힌트: {hints}")
    print()

[국방과학연구소가 파일 전송하는 시스템을 크게 개선하려는 사업 예산은?]
  힌트: ['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '국방과학연구소_대용량 자료전송시스템 고도화.hwp']

[국방과학연구소가 문서나 자료를 체계적으로 관리하고 보안을 강화하려는 사업 예산은?]
  힌트: ['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '국방과학연구소_대용량 자료전송시스템 고도화.hwp']

[축산물품질평가원이 동물 이력을 추적하는 시스템을 손보는 사업 기간은?]
  힌트: ['축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp', '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp']

[축산물품질평가원의 벌꿀 등급 판정 관련 시스템 개선 사업 예산은?]
  힌트: ['축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp', '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp']



In [46]:
verify_answers_3 = [
    "국방과학연구소가 파일 전송하는 시스템을 크게 개선하려는 사업 예산은?",
    "국방과학연구소가 문서나 자료를 체계적으로 관리하고 보안을 강화하려는 사업 예산은?",
    "축산물품질평가원이 동물 이력을 추적하는 시스템을 손보는 사업 기간은?",
    "축산물품질평가원의 벌꿀 등급 판정 관련 시스템 개선 사업 예산은?",
]

for q in verify_answers_3:
    print(f"{q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

국방과학연구소가 파일 전송하는 시스템을 크게 개선하려는 사업 예산은?
316,800,000원. 근거: 국방과학연구소_대용량 자료전송시스템 고도화.hwp

국방과학연구소가 문서나 자료를 체계적으로 관리하고 보안을 강화하려는 사업 예산은?
282,300,000원 — 국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp

축산물품질평가원이 동물 이력을 추적하는 시스템을 손보는 사업 기간은?
계약일로부터 6개월입니다. 근거: 문서 '축산물품질평가원_축산물이력관리시스템 개선(정보화 사업).hwp'

축산물품질평가원의 벌꿀 등급 판정 관련 시스템 개선 사업 예산은?
49,000,000원 — 축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp

